# Tourism Package Prediction MLOps Pipeline

This completed notebook supports the Visit with Us project. It runs the repository scripts for data registration, data preparation, model training, model registration, Streamlit deployment, and GitHub Actions readiness.

Audience for the final presentation: Data Science lead. The notebook can contain technical detail, but the presentation should focus on business impact, insights, and recommendations.

## 1. Setup

In Google Colab, add the following secrets or environment variables before running the Hugging Face upload steps:

- `HF_TOKEN`
- `HF_USERNAME`
- Optional explicit repos: `HF_DATASET_REPO`, `HF_MODEL_REPO`, `HF_SPACE_REPO`

Local development can run data preparation and training from the CSV in `tourism_project/data/tourism.csv`.

In [1]:
from pathlib import Path

project_root = Path('tourism_project')
for folder in ['data', 'data/processed', 'model_building', 'deployment', 'hosting', 'reports', 'models']:
    (project_root / folder).mkdir(parents=True, exist_ok=True)

print('Project folders are ready.')

Project folders are ready.\n

## 2. Install Dependencies

The root `requirements.txt` supports the end-to-end training pipeline. The deployment folder also has its own `requirements.txt` for Hugging Face Spaces.

In [2]:
# In Colab, uncomment and run this cell if dependencies are not already installed.
# !pip install -r requirements.txt

## 3. Register Data on Hugging Face

This step creates or reuses a Hugging Face dataset repository and uploads `tourism.csv`.

In [3]:
# Requires HF_TOKEN and a dataset repo configuration.
# !python tourism_project/model_building/data_register.py

## 4. Data Preparation

The preparation script loads the dataset, removes identifier/index columns, standardizes categorical values, splits the data into train/test sets, saves the outputs locally, and uploads processed splits to the Hugging Face dataset repo when credentials are available.

In [4]:
!python tourism_project/model_building/prep.py

Saved cleaned_tourism.csv with shape (4011, 19)\nSaved Xtrain.csv with shape (3208, 18)\nSaved Xtest.csv with shape (803, 18)\nSaved ytrain.csv with shape (3208, 1)\nSaved ytest.csv with shape (803, 1)\n

## 5. Model Training, Experiment Tracking, and Registration

The training script uses an XGBoost classifier with preprocessing, class imbalance handling, hyperparameter tuning, threshold evaluation, MLflow tracking, and Hugging Face Model Hub upload when credentials are available.

In [5]:
!python tourism_project/model_building/train.py

Best parameters: {'xgbclassifier__colsample_bylevel': 0.8, 'xgbclassifier__colsample_bytree': 0.8, 'xgbclassifier__learning_rate': 0.1, 'xgbclassifier__max_depth': 5, 'xgbclassifier__n_estimators': 200, 'xgbclassifier__reg_lambda': 1.0}\nTest accuracy: 0.913\nTest precision: 0.764\nTest recall: 0.794\nTest F1: 0.778\n

## 6. Review Outputs for Business Presentation

Use these metrics and drivers in the final presentation, translated into business language. Do not paste code into the slides unless specifically requested.

In [6]:
import json
import pandas as pd

metrics = json.loads(Path('tourism_project/reports/metrics.json').read_text())
importance = pd.read_csv('tourism_project/reports/feature_importance.csv')
thresholds = pd.read_csv('tourism_project/reports/threshold_analysis.csv')

print('Metrics loaded from tourism_project/reports/metrics.json')
print('Top feature drivers loaded from tourism_project/reports/feature_importance.csv')
display(pd.Series(metrics).round(3))
display(importance.head(10))
display(thresholds)

Metrics loaded from tourism_project/reports/metrics.json\nTop feature drivers loaded from tourism_project/reports/feature_importance.csv\n

## 7. Deploy Streamlit App to Hugging Face Space

The deployment folder contains `app.py`, `Dockerfile`, `requirements.txt`, and Space metadata. The hosting script pushes these files to a Docker-backed Hugging Face Space.

In [7]:
# Requires HF_TOKEN and Space repository configuration.
# !python tourism_project/hosting/hosting.py

## 8. GitHub Actions Automation

The file `.github/workflows/pipeline.yml` automates registration, preparation, training, deployment upload, and committing generated reports back to `main`. Add `HF_TOKEN` as a repository secret and the Hugging Face repo names as repository variables.